[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://drive.google.com/file/d/13PkpkDbjLCrttyEV42UBihobtuiKDAO-/view?usp=drive_link)

# LLM Evaluation – Partial Dataset

This notebook demonstrates how to evaluate when you have only questions (no answers yet). Floeval generates `llm_response` for each sample at runtime, then runs the metrics on the generated responses.

**Objectives**
- Install Floeval and configure credentials
- Build a partial dataset (samples without `llm_response`)
- Set `dataset_generator_model` so Floeval generates responses
- Run evaluation and inspect results

## 1. Installation

Install Floeval before running this notebook.

In [1]:
%pip install git+https://github.com/FloTorch/floeval.git@dev

Note: you may need to restart the kernel to use updated packages.


c:\Users\FL_LPT-812\Documents-SELF\Projects\Floeval\.venv\Scripts\python.exe: No module named pip


## 2. Configuration Constants

Set the following constants before running. Replace placeholder values with your API credentials and model identifiers.

**Provider flexibility:** You can use any OpenAI-compatible provider (OpenAI, Azure OpenAI, Anthropic, local models, etc.) — set the appropriate `base_url` and model names for your provider.

**Using FloTorch:** If you want to use FloTorch keys and gateway, obtain credentials from the [FloTorch Console](https://docs.flotorch.cloud/introduction/).

In [2]:
import getpass

LLM_BASE_URL = "https://api.openai.com/v1"
API_KEY = getpass.getpass("your-api-key")
CHAT_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

In [3]:
import os
import dotenv

dotenv.load_dotenv(dotenv_path=".env.notebooktesting")


# Optional: override constants from environment variables.
LLM_BASE_URL = os.environ["LLM_BASE_URL"]
API_KEY = os.environ["API_KEY"]
CHAT_MODEL = os.environ["CHAT_MODEL"]
EMBEDDING_MODEL = os.environ["EMBEDDING_MODEL"]

## 3. Imports

The following cell imports the evaluation components and the LLM configuration schema.

In [ ]:
from floeval import DatasetLoader, Evaluation
from floeval.config.schemas.io.llm import OpenAIProviderConfig


## 4. Configure the LLM

The LLM configuration is built using the constants defined above. It is used for both response generation and evaluation.

In [ ]:
llm_config = OpenAIProviderConfig(
    base_url=LLM_BASE_URL,
    api_key=API_KEY,
    chat_model=CHAT_MODEL,
    embedding_model=EMBEDDING_MODEL,
)

## 5. Load the Partial Dataset

A partial dataset is created with `user_input` only. The `partial_dataset=True` flag indicates that `llm_response` will be generated by Floeval at runtime.

In [ ]:
partial_dataset = DatasetLoader.from_samples(
    [
        {"user_input": "What is Python?"},
        {"user_input": "What is RAG?"},
    ],
    partial_dataset=True,
)
print(f"Partial dataset loaded: {len(partial_dataset.samples)} samples (no llm_response)")

Partial dataset loaded: 2 samples (no llm_response)


In [ ]:
from floeval.config.schemas.io.dataset import PartialDataset
from floeval.core.execution.llm_executor import OpenAIProvider
from floeval.core.execution.response_synthesizer import apopulate_llm_responses

llm_provider = OpenAIProvider(llm_config)

assert isinstance(partial_dataset, PartialDataset)
complete_dataset = await apopulate_llm_responses(partial_dataset, llm_provider)

# test print the samples in complete dataset
for sample in complete_dataset.samples:
    print(f"User Input: {sample.user_input}")
    print(f"LLM Response: {sample.llm_response[:80]}...")  # print first 80 chars of the response
    print("-" * 50)

User Input: What is Python?
LLM Response: Okay, let's break down what Python is. In simple terms, Python is a **high-level...
--------------------------------------------------
User Input: What is RAG?
LLM Response: Okay, let's break down RAG – Retrieval-Augmented Generation. It's a really hot t...
--------------------------------------------------


In [ ]:
from floeval.core.execution.response_synthesizer import populate_llm_responses

assert isinstance(partial_dataset, PartialDataset)
complete_dataset = populate_llm_responses(partial_dataset, llm_provider)

# test print the samples in complete dataset
for sample in complete_dataset.samples:
    print(f"User Input: {sample.user_input}")
    print(f"LLM Response: {sample.llm_response[:80]}...")  # print first 80 chars of the response
    print("-" * 50)

User Input: What is Python?
LLM Response: Okay, let's break down what Python is. In simple terms, Python is a **high-level...
--------------------------------------------------
User Input: What is RAG?
LLM Response: Okay, let's break down RAG – Retrieval-Augmented Generation. It's a really hot t...
--------------------------------------------------


## 6. Create and Run the Evaluation

The `dataset_generator_model` parameter specifies which model Floeval uses to generate responses. For each sample, Floeval sends `user_input` to the LLM, stores the response, and then runs the metrics on the generated output.

In [12]:
evaluation = Evaluation(
    dataset=partial_dataset,
    llm_config=llm_config,
    metrics=["answer_relevancy"],
    default_provider="ragas",
    # NOTE: dataset_generator_model needed to populate llm_responses for the partial dataset
    dataset_generator_model=CHAT_MODEL,
)

results = evaluation.run()
print("Aggregate scores:", results.aggregate_scores)

Error computing answer relevancy: asyncio.run() cannot be called from a running event loop
Traceback (most recent call last):
  File "C:\Users\FL_LPT-812\Documents-SELF\Projects\Floeval\floeval\metric_providers\ragas\metrics.py", line 146, in evaluate
    score = asyncio.run(self.ragas_metric.single_turn_ascore(ragas_sample))
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\FL_LPT-812\AppData\Local\Programs\Python\Python312\Lib\asyncio\runners.py", line 191, in run
    raise RuntimeError(
RuntimeError: asyncio.run() cannot be called from a running event loop
C:\Users\FL_LPT-812\Documents-SELF\Projects\Floeval\floeval\metric_providers\ragas\metrics.py:154: RuntimeWarning: coroutine 'SingleTurnMetric.single_turn_ascore' was never awaited
  return MetricResult(
Error computing answer relevancy: asyncio.run() cannot be called from a running event loop
Traceback (most recent call last):
  File "C:\Users\FL_LPT-812\Documents-SELF\Projects\Floeval\f

Aggregate scores: {}


## 7. Inspect Generated Responses

Each sample result includes the generated `llm_response` and the metric scores. This enables inspection of both the generated text and the evaluation scores per sample.

In [23]:
for i, sr in enumerate(results.sample_results, start=1):
    print(f"Sample {i}: {sr['user_input']}")
    print(f"  Generated: {sr.get('llm_response', '')[:80]}...")
    for key, data in sr.get("metrics", {}).items():
        print(f"  {key}: {data.get('score')}")

Sample 1: What is Python?
  Generated: Okay, let's break down what Python is. In simple terms, Python is a **high-level...
  ragas:answer_relevancy: 0.0
Sample 2: What is RAG?
  Generated: Okay, let's break down RAG – Retrieval-Augmented Generation. It's a really hot t...
  ragas:answer_relevancy: 0.7447350376623324


## Summary

This notebook demonstrated the evaluation of LLM responses using a partial dataset with Floeval.

The key components included:

1. **Partial Dataset Loading**: A dataset with `user_input` only was loaded using `partial_dataset=True`.
2. **LLM Configuration**: The OpenAI-compatible provider was configured for both generation and evaluation.
3. **Response Generation**: The `dataset_generator_model` parameter enabled Floeval to generate responses at runtime before scoring.
4. **Evaluation Execution**: The `answer_relevancy` metric was run on the generated responses.
5. **Results Inspection**: Generated text and per-sample scores were accessed through `results.sample_results`.

This example showcases the workflow for evaluating when you have questions but no pre-generated answers.